# Assignment 02: Logistic Regression and Evaluation Metrics

**Module:** 01 — Classical ML Foundations  
**Estimated Time:** 8-10 hours  
**Difficulty:** Intermediate to Advanced

---

## Learning Objectives

- Implement logistic regression from scratch with gradient descent
- Derive binary cross-entropy from maximum likelihood estimation
- Build confusion matrix, precision, recall, F1, ROC curve, and AUC from scratch
- Implement softmax for multi-class classification

### Key Equations

**Sigmoid:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Binary Cross-Entropy Loss:**
$$L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

**Gradients:**
$$\frac{\partial L}{\partial \mathbf{w}} = \frac{1}{n} \mathbf{X}^T (\sigma(\mathbf{X}\mathbf{w} + b) - \mathbf{y}), \quad \frac{\partial L}{\partial b} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)$$

**Softmax:**
$$\text{softmax}(z_j) = \frac{e^{z_j}}{\sum_{k} e^{z_k}}$$

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, make_circles, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import (
    confusion_matrix as sklearn_cm,
    precision_score, recall_score, f1_score as sklearn_f1,
    roc_curve as sklearn_roc, roc_auc_score
)

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Setup complete.')

---
## Part 1A: The Sigmoid Function (5 points)

Implement a numerically stable sigmoid:
- For $z \geq 0$: $\sigma(z) = \frac{1}{1 + e^{-z}}$
- For $z < 0$: $\sigma(z) = \frac{e^z}{1 + e^z}$

The derivative satisfies: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """Compute the sigmoid function (numerically stable).

    Args:
        z: Input array of any shape

    Returns:
        Array of same shape with values in (0, 1)
    """
    # YOUR CODE HERE
    # Use np.where for numerical stability:
    #   z >= 0: 1 / (1 + exp(-z))
    #   z < 0:  exp(z) / (1 + exp(z))
    pass


def sigmoid_derivative(z: np.ndarray) -> np.ndarray:
    """Compute sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))."""
    s = sigmoid(z)
    return s * (1 - s)

In [ ]:
# --- Test Part 1A ---
# Numerical stability test
assert not np.isnan(sigmoid(np.array([-1000.0]))), 'sigmoid(-1000) should not be NaN'
assert not np.isnan(sigmoid(np.array([1000.0]))), 'sigmoid(1000) should not be NaN'
assert not np.isinf(sigmoid(np.array([-1000.0]))), 'sigmoid(-1000) should not be Inf'
assert not np.isinf(sigmoid(np.array([1000.0]))), 'sigmoid(1000) should not be Inf'
assert np.isclose(sigmoid(np.array([0.0]))[0], 0.5), 'sigmoid(0) should be 0.5'

# Derivative verification via finite differences
z_test = np.array([0.0, 1.0, -1.0, 5.0, -5.0])
h = 1e-7
numerical_deriv = (sigmoid(z_test + h) - sigmoid(z_test - h)) / (2 * h)
analytical_deriv = sigmoid_derivative(z_test)
assert np.allclose(numerical_deriv, analytical_deriv, atol=1e-5), \
    'Analytical derivative should match numerical'

# Plot sigmoid
z_plot = np.linspace(-10, 10, 200)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_plot, sigmoid(z_plot), label='$\\sigma(z)$', linewidth=2)
ax.plot(z_plot, sigmoid_derivative(z_plot), label="$\\sigma'(z)$", linewidth=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('z')
ax.set_ylabel('Value')
ax.set_title('Sigmoid Function and Its Derivative')
ax.legend()
plt.tight_layout()
plt.show()

print('Part 1A tests passed!')

---
## Part 1B: Cross-Entropy Loss (10 points)

### MLE Derivation (write in this cell)

**Step 1 — Likelihood of a single Bernoulli sample:**
$$P(y_i | \hat{y}_i) = \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

**Step 2 — Likelihood of $n$ independent samples:**
$$\mathcal{L} = \prod_{i=1}^{n} \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

**Step 3 — Log-likelihood:**
$$\log \mathcal{L} = \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

**Step 4 — Negate and normalize:**
$$L = -\frac{1}{n} \log \mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

In [ ]:
def binary_cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute binary cross-entropy loss.

    L = -(1/n) * sum(y*log(y_hat) + (1-y)*log(1-y_hat))

    Args:
        y_true: True labels, shape (n,)
        y_pred: Predicted probabilities, shape (n,)

    Returns:
        Scalar loss value
    """
    # YOUR CODE HERE
    # Clip y_pred to [epsilon, 1 - epsilon] to avoid log(0)
    # epsilon = 1e-15
    pass

In [ ]:
# --- Test Part 1B ---
y_t = np.array([1, 0, 1, 1, 0])
y_p = np.array([0.9, 0.1, 0.8, 0.7, 0.3])
loss = binary_cross_entropy(y_t, y_p)
print(f'BCE loss: {loss:.6f}')
assert loss > 0, 'Loss should be positive'
assert not np.isnan(loss), 'Loss should not be NaN'

# Edge case: perfect predictions should give very low loss
loss_perfect = binary_cross_entropy(y_t, np.array([1.0, 0.0, 1.0, 1.0, 0.0]))
assert loss_perfect < loss, 'Perfect predictions should have lower loss'
print('Part 1B tests passed!')

---
## Part 1C: Logistic Regression Classifier (20 points)

Implement the full classifier with gradient descent training.

In [ ]:
class LogisticRegression:
    """Logistic regression classifier trained via gradient descent."""

    def __init__(self, learning_rate: float = 0.01, n_iterations: int = 1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights: np.ndarray = None  # shape: (n_features,)
        self.bias: float = None
        self.loss_history: list[float] = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LogisticRegression':
        """Train using gradient descent on binary cross-entropy.

        Gradients:
            dw = (1/n) * X^T @ (sigmoid(X @ w + b) - y)
            db = (1/n) * sum(sigmoid(X @ w + b) - y)

        Args:
            X: shape (n_samples, n_features)
            y: shape (n_samples,) with values in {0, 1}
        """
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.n_iterations):
            # YOUR CODE HERE
            # 1. Compute predictions: y_pred = sigmoid(X @ w + b)
            # 2. Compute BCE loss and append to loss_history
            # 3. Compute gradients
            # 4. Update weights and bias
            pass

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return probability estimates.

        Returns:
            Probabilities, shape (n_samples,)
        """
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Return binary predictions.

        Returns:
            Binary predictions, shape (n_samples,)
        """
        # YOUR CODE HERE
        pass

In [ ]:
# --- Test Part 1C ---
X_cls, y_cls = make_classification(n_samples=500, n_features=2, n_redundant=0,
                                   n_informative=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2,
                                                    random_state=42)

model_lr = LogisticRegression(learning_rate=0.1, n_iterations=1000)
model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f'Accuracy: {accuracy:.4f}')

# Sanity check against sklearn
sk_model = SklearnLR(max_iter=1000)
sk_model.fit(X_train, y_train)
sk_acc = sk_model.score(X_test, y_test)
print(f'sklearn accuracy: {sk_acc:.4f}')

assert accuracy > 0.7, 'Accuracy should be > 70%'
assert len(model_lr.loss_history) == 1000, 'Should have 1000 loss values'
assert model_lr.loss_history[-1] < model_lr.loss_history[0], 'Loss should decrease'

# Plot loss curve
fig, ax = plt.subplots()
ax.plot(model_lr.loss_history)
ax.set_xlabel('Iteration')
ax.set_ylabel('BCE Loss')
ax.set_title('Logistic Regression Training Loss')
plt.tight_layout()
plt.show()

print('Part 1C tests passed!')

---
## Part 2A: Linear Decision Boundary (5 points)

The decision boundary is where $\sigma(\mathbf{x}^T \mathbf{w} + b) = 0.5$, i.e., $\mathbf{x}^T \mathbf{w} + b = 0$.

For 2D: $x_2 = -(w_1 x_1 + b) / w_2$

In [ ]:
def plot_decision_boundary(model, X: np.ndarray, y: np.ndarray,
                           title: str = 'Decision Boundary', ax=None):
    """Plot data points and the decision boundary for a 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # Get predictions on grid
    if hasattr(model, 'predict_proba'):
        Z = model.predict_proba(grid).reshape(xx.shape)
    else:
        Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', label='Class 0',
               edgecolors='k', alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Class 1',
               edgecolors='k', alpha=0.7)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(title)
    ax.legend()
    return ax

In [ ]:
# --- Part 2A: Plot linear decision boundary ---
# YOUR CODE HERE
# plot_decision_boundary(model_lr, X_cls, y_cls, 'Linear Decision Boundary')
pass

---
## Part 2B: Nonlinear Decision Boundaries (10 points)

Create a non-linearly separable dataset, show logistic regression fails, then add polynomial features (degree 2 and 3) to fix it.

In [ ]:
# --- Part 2B: Nonlinear Boundaries ---
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=42)

# YOUR CODE HERE
# 1. Train logistic regression on raw data, plot decision boundary (linear - fails)
# 2. Add polynomial features degree 2, retrain, plot (should curve)
# 3. Add polynomial features degree 3, retrain, plot (should curve more)
# Use fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pass

**Why do polynomial features allow logistic regression to learn nonlinear boundaries?**

*YOUR ANSWER HERE*

---
## Part 3A: Confusion Matrix (5 points)

In [ ]:
def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Compute the confusion matrix.

    Returns:
        2x2 numpy array: [[TN, FP], [FN, TP]]
    """
    # YOUR CODE HERE
    pass


def plot_confusion_matrix(cm: np.ndarray, ax=None):
    """Display a confusion matrix as a heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Predicted Negative', 'Predicted Positive'])
    ax.set_yticklabels(['Actual Negative', 'Actual Positive'])
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{labels[i][j]}\n{cm[i, j]}',
                    ha='center', va='center', fontsize=14)
    ax.set_title('Confusion Matrix')
    plt.colorbar(im, ax=ax)
    return ax

In [ ]:
# --- Test Part 3A ---
y_pred_test = model_lr.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test)
cm_sklearn = sklearn_cm(y_test, y_pred_test)

print('Your CM:\n', cm)
print('sklearn CM:\n', cm_sklearn)
assert np.array_equal(cm, cm_sklearn), 'Should match sklearn'

plot_confusion_matrix(cm)
plt.tight_layout()
plt.show()
print('Part 3A tests passed!')

---
## Part 3B: Precision, Recall, and F1 (10 points)

In [ ]:
def precision(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Precision = TP / (TP + FP).
    Of all predicted positive, how many are truly positive?
    """
    # YOUR CODE HERE
    pass


def recall(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Recall = TP / (TP + FN).
    Of all truly positive, how many did we catch?
    """
    # YOUR CODE HERE
    pass


def f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """F1 = 2 * (precision * recall) / (precision + recall).
    Harmonic mean of precision and recall.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Part 3B ---
p = precision(y_test, y_pred_test)
r = recall(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print(f'Precision: {p:.4f} (sklearn: {precision_score(y_test, y_pred_test):.4f})')
print(f'Recall:    {r:.4f} (sklearn: {recall_score(y_test, y_pred_test):.4f})')
print(f'F1:        {f1:.4f} (sklearn: {sklearn_f1(y_test, y_pred_test):.4f})')

assert abs(p - precision_score(y_test, y_pred_test)) < 1e-10, 'Precision should match'
assert abs(r - recall_score(y_test, y_pred_test)) < 1e-10, 'Recall should match'
assert abs(f1 - sklearn_f1(y_test, y_pred_test)) < 1e-10, 'F1 should match'
print('Part 3B tests passed!')

In [ ]:
# --- Part 3B: Imbalanced scenario ---
# YOUR CODE HERE
# 1. Create imbalanced data where accuracy is high but precision is low
#    (e.g., 95% class 0, 5% class 1)
# 2. Create scenario where precision is high but recall is low
#    (use a high threshold like 0.9)
# 3. Print accuracy, precision, recall for each
pass

**When would you prioritize precision over recall? Recall over precision?**

*YOUR ANSWER HERE*

---
## Part 3C: ROC Curve and AUC (15 points)

In [ ]:
def roc_curve(y_true: np.ndarray, y_scores: np.ndarray):
    """Compute the ROC curve.

    Args:
        y_true: True binary labels, shape (n,)
        y_scores: Predicted probabilities, shape (n,)

    Returns:
        fpr: Array of false positive rates
        tpr: Array of true positive rates
        thresholds: Array of thresholds used
    """
    # YOUR CODE HERE
    # 1. Get sorted unique thresholds (descending)
    # 2. For each threshold, compute FPR = FP/(FP+TN) and TPR = TP/(TP+FN)
    # 3. Return arrays of fpr, tpr, thresholds
    pass


def auc(fpr: np.ndarray, tpr: np.ndarray) -> float:
    """Compute Area Under the ROC Curve using the trapezoidal rule."""
    # YOUR CODE HERE
    # Use np.trapz or implement trapezoidal rule manually
    pass

In [ ]:
# --- Test Part 3C ---
y_scores = model_lr.predict_proba(X_test)

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc_val = auc(fpr, tpr)

fpr_sk, tpr_sk, _ = sklearn_roc(y_test, y_scores)
auc_sk = roc_auc_score(y_test, y_scores)

print(f'Your AUC:    {auc_val:.4f}')
print(f'sklearn AUC: {auc_sk:.4f}')

# Plot ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, 'b-', label=f'Your ROC (AUC={auc_val:.3f})', linewidth=2)
ax.plot(fpr_sk, tpr_sk, 'r--', label=f'sklearn ROC (AUC={auc_sk:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.show()
print('Part 3C tests passed!')

In [ ]:
# --- Part 3C: Three ROC curves (good, mediocre, random) ---
# YOUR CODE HERE
# 1. Good classifier: your trained model
# 2. Mediocre classifier: train on noisy features
# 3. Random classifier: random predictions
# Plot all three on the same figure
pass

**What does AUC = 0.5, AUC = 1.0, and AUC = 0.0 mean?**

*YOUR ANSWER HERE*

---
## Part 4A: Softmax Implementation (5 points)

$$\text{softmax}(z_j) = \frac{e^{z_j - \max(z)}}{\sum_k e^{z_k - \max(z)}}$$

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    """Compute softmax (numerically stable).

    Args:
        z: shape (n_samples, n_classes)

    Returns:
        Probabilities, shape (n_samples, n_classes), each row sums to 1
    """
    # YOUR CODE HERE
    # 1. Subtract max per row for numerical stability
    # 2. Exponentiate
    # 3. Normalize each row
    pass

In [ ]:
# --- Test Part 4A ---
z_test = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0], [1000.0, 0.0, 0.0]])
probs = softmax(z_test)

print('Softmax output:')
print(probs)
print('Row sums:', probs.sum(axis=1))

assert np.allclose(probs.sum(axis=1), 1.0), 'Each row must sum to 1'
assert not np.any(np.isnan(probs)), 'No NaN values'
assert not np.any(np.isinf(probs)), 'No Inf values'

# Verify 2-class softmax is equivalent to sigmoid
z_2class = np.array([[2.0, 0.0], [-1.0, 0.0]])
sm_probs = softmax(z_2class)[:, 0]
sig_probs = sigmoid(z_2class[:, 0] - z_2class[:, 1])
assert np.allclose(sm_probs, sig_probs, atol=1e-6), \
    'For 2 classes, softmax should be equivalent to sigmoid'
print('Part 4A tests passed!')

---
## Part 4B: Multi-Class Logistic Regression (5 points)

Extend logistic regression to multi-class using softmax + categorical cross-entropy, or implement one-vs-rest.

In [ ]:
# --- Part 4B: Multi-Class ---
# YOUR CODE HERE
# Option A: softmax + categorical cross-entropy
# Option B: one-vs-rest with your binary logistic regression
#
# Train on Iris dataset or a multi-class subset
# Evaluate with multi-class confusion matrix and per-class precision/recall
pass

---
## Part 5: Written Analysis (10 points)

### Q1: Why does cross-entropy work better than MSE for classification?

*YOUR ANSWER HERE*

### Q2: Why is the ROC curve more informative than a single accuracy number?

*YOUR ANSWER HERE*

### Q3: Why softmax instead of just normalizing outputs to sum to 1?

*YOUR ANSWER HERE*

### Q4: How does a neural network learn nonlinear boundaries differently from polynomial features?

*YOUR ANSWER HERE*

### Q5: For medical diagnosis, which metric would you optimize and why?

*YOUR ANSWER HERE*